# Mandarin Tone Discrimination Task Analysis

In [ ]:
import importlib
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchaudio.transforms import Resample
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import sys
sys.path.append('../')

import robustness.audio_functions.audio_transforms as at
from lightning_scripts.utils import model_build_utils
importlib.reload(model_build_utils)
from lightning_scripts.utils.model_build_utils import get_model

import figure_utils
from importlib import reload
reload(figure_utils)
from figure_utils import (
    normalize_model_name,
    normalize_model_list,
    normalize_palette_dict,
    marker_palette_for_hue,
    build_model_palette,
    get_standard_hue_order,
    get_standard_base_colors,
    get_model_groups,
    get_plot_groups,
    model_label,
    plot_grouped_bars,
)

torch.set_float32_matmul_precision("medium")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline

## Dataset Setup

In [ ]:
tone_perfect_dir = Path("~/ceph/datasets/tone_perfect").expanduser()
tone_perfect_SR = 44100
model_sr = 20_000

def load_audio(rel_path, tone_perfect_dir=tone_perfect_dir):
    audio_path = tone_perfect_dir / rel_path
    audio, sr = torchaudio.load(str(audio_path))
    return audio, sr


class TonePerfectDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        tone_perfect_dir,
        resample_sr=20_000,
        pair_mode=False,
        include_negative=True,
        allow_tone1=False,
    ):
        tp_metadata = pd.read_csv(tone_perfect_dir / "audio_metadata.csv")
        if not allow_tone1:
            tp_metadata = tp_metadata[tp_metadata.tone != 1]
        self.metadata = tp_metadata
        self.talker_ids = tp_metadata.speaker.unique()
        self.base_syllables = tp_metadata.base_syllable.unique()
        self.tones = tp_metadata.tone.unique()
        self.pair_mode = pair_mode
        self.include_negative = include_negative
        if resample_sr is not None:
            self.resamp = Resample(tone_perfect_SR, resample_sr)
        else:
            self.resamp = None

        # cache speaker/tone combos that have at least two base syllables
        self.valid_pair_keys = [
            (spk, tone)
            for (spk, tone), df in self.metadata.groupby(["speaker", "tone"])
            if df.base_syllable.nunique() >= 2
        ]

    def __len__(self):
        return len(self.metadata)

    def _load_and_resample(self, file_name):
        audio, _ = load_audio(file_name)
        audio = audio[0]  # mono
        if self.resamp is not None:
            audio = self.resamp(audio)
        return audio

    def __getitem__(self, idx):
        rng = np.random.RandomState(idx)

        if self.pair_mode:
            if len(self.valid_pair_keys) == 0:
                raise RuntimeError(
                    "No speaker/tone combinations with >=2 base syllables available."
                )

            # pick a talker/tone with at least two base syllables
            talker_id, tone = self.valid_pair_keys[
                rng.randint(0, len(self.valid_pair_keys))
            ]
            examples = self.metadata[
                (self.metadata.speaker == talker_id) & (self.metadata.tone == tone)
            ]
            base_choices = examples.base_syllable.unique()
            if len(base_choices) < 2:
                raise RuntimeError(
                    f"Insufficient base syllable variety for speaker {talker_id}, tone {tone}."
                )

            # robustly sample two bases that actually have rows
            max_tries = 10
            for _ in range(max_tries):
                chosen_bases = rng.choice(base_choices, size=2, replace=False)
                pair_audio, pair_syllables = [], []
                ok = True
                for base in chosen_bases:
                    base_examples = examples[examples.base_syllable == base]
                    if len(base_examples) == 0:
                        ok = False
                        break
                    row = base_examples.sample(random_state=rng).iloc[0]
                    pair_audio.append(self._load_and_resample(row.file_name))
                    pair_syllables.append(row.syllable)
                if ok:
                    break
            else:
                raise RuntimeError("Failed to sample two valid base syllables for this speaker/tone")

            negative_audio = None
            negative_meta = None
            if self.include_negative:
                neg_candidates = self.metadata[
                    (self.metadata.speaker == talker_id) & (self.metadata.tone != tone)
                ]
                if len(neg_candidates) > 0:
                    neg_row = neg_candidates.sample(random_state=rng).iloc[0]
                    negative_audio = self._load_and_resample(neg_row.file_name)
                    negative_meta = neg_row

            return {
                "audio_pair": pair_audio,
                "pair_syllables": pair_syllables,
                "pair_tone": tone,
                "speaker": talker_id,
                "negative_audio": negative_audio,
                "negative_meta": negative_meta,
            }

        # Original triplet sampling for tones 2/3/4 from same base syllable
        talker_id = rng.choice(self.talker_ids)
        base_syllable = rng.choice(self.base_syllables)
        examples = self.metadata[
            (self.metadata.speaker == talker_id)
            & (self.metadata.base_syllable == base_syllable)
        ]
        examples = examples.sort_values(by="tone")
        tone_audio = []
        syllables = examples.syllable.values
        for ix in range(min(3, len(examples))):
            tone_audio.append(self._load_and_resample(examples.iloc[ix].file_name))

        return {"audio": tone_audio, "syllables": syllables}